In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/mersad/protBuild")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from ml.src.training.dataset.protein_dataset import ProteinTextDataset, make_collate_fn
from ml.src.training.trainer.train import train_model_protein2text
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, AutoConfig
from ml.src.models.protein2Text.prot2text import Prot2Text
from torch.utils.data import DataLoader
import pandas as pd
import random
import torch
import json


In [3]:
pretrained_protein_model_name = 'facebook/esm2_t6_8M_UR50D'
pretrained_text_model_name = 'openai-community/gpt2'


In [4]:
protein_tokenizer = AutoTokenizer.from_pretrained(pretrained_protein_model_name, trust_remote_code=True)
text_tokenizer = AutoTokenizer.from_pretrained(pretrained_text_model_name, trust_remote_code=True)

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
protein_model = AutoModel.from_pretrained(pretrained_protein_model_name, trust_remote_code=True).to(device)

text_model_config = AutoConfig.from_pretrained(pretrained_text_model_name, add_cross_attention=True)
text_model = AutoModelForCausalLM.from_pretrained(pretrained_text_model_name, trust_remote_code=True, config=text_model_config).to(device)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                                                 | Status  | 
----------------------------------------------------+---------+-
transformer.h.{0...11}.ln_cross_attn.weight         | MISSING | 
transformer.h.{0...11}.crossattention.c_attn.weight | MISSING | 
transformer.h.{0...11}.crossattention.q_attn.bias   | MISSING | 
transformer.h.{0...11}.crossattention.c_proj.weight | MISSING | 
transformer.h.{0...11}.crossattention.c_proj.bias   | MISSING | 
transformer.h.{0...11}.ln_cross_attn.bias           | MISSING | 
transformer.h.{0...11}.crossattention.q_attn.weight | MISSING | 
transformer.h.{0...11}.crossattention.c_attn.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
protein_model.config.max_position_embeddings


1026

In [15]:
text_model.config.max_position_embeddings

1024

In [16]:
protein_tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token']

In [17]:
text_tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token']

In [18]:
print(protein_tokenizer.pad_token_id)
print(protein_tokenizer.pad_token)

print(text_tokenizer.pad_token_id)
print(text_tokenizer.pad_token)

1
<pad>
None
None


In [6]:
protein_pad_id = protein_tokenizer.pad_token_id
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
text_pad_id = text_tokenizer.pad_token_id

In [7]:
train_dataset = ProteinTextDataset(
    PROJECT_ROOT / "ml/datasets/prot2text_train.jsonl",
    protein_tokenizer,
    text_tokenizer,
    128,
    64,
)


val_dataset = ProteinTextDataset(
    PROJECT_ROOT / "ml/datasets/prot2text_val.jsonl",
    protein_tokenizer,
    text_tokenizer,
    128,
    64,   
)

test_dataset = ProteinTextDataset(
    PROJECT_ROOT / "ml/datasets/prot2text_test.jsonl",
    protein_tokenizer,
    text_tokenizer,
    128,
    64,   
)

In [8]:
collate_fn = make_collate_fn(
    protein_pad_id=protein_pad_id,
    text_pad_id=text_pad_id,
    ignore_index=-100,
)

In [9]:


train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    collate_fn=collate_fn,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    collate_fn=collate_fn,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    collate_fn=collate_fn,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
)

In [10]:
input_dim = protein_model.config.hidden_size
output_dim = text_model.config.hidden_size

In [11]:

model = Prot2Text(protein_model, text_model, 
                  input_dim=input_dim, hidden_dim=output_dim*2,
                    output_dim=output_dim, dropout=0.1,
                      bias=False)

In [ ]:
for param in model.protein_model.parameters():
    param.requires_grad = False

trainable_params = [
    param for param in model.parameters()
    if param.requires_grad
]

print("Trainable parameters:",
      sum(p.numel() for p in trainable_params))

Trainable parameters: 154478592


In [30]:
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=0.01,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200_000 * 3,
    eta_min=1e-6,
)

In [31]:
train_losses, val_losses = train_model_protein2text(model, train_loader, val_loader, optimizer, scheduler, device, 5, 100000, 1000)

Epoch 1/5: 0batch [00:00, ?batch/s][transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Epoch 1/5: 5086batch [14:02,  6.03batch/s, loss=1.739][W930 20:15:12.811598072 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2097152 bytes (free: 31522816, total: 3953721344).
Epoch 1/5: 100001batch [4:37:50,  5.08s/batch, loss=0.983]

Epoch 1 (Step 100000):
Train loss 0.790, Val loss 0.503
Train perplexity 2.204, Val perplexity 1.653


Epoch 1/5: 105407batch [4:52:45,  6.00batch/s, loss=0.049]
Epoch 2/5: 94594batch [4:21:00,  4.82s/batch, loss=0.383]

Epoch 2 (Step 200000):
Train loss 0.463, Val loss 0.382
Train perplexity 1.589, Val perplexity 1.465


Epoch 2/5: 105407batch [4:50:49,  6.04batch/s, loss=0.027]
Epoch 3/5: 89187batch [4:05:57,  4.71s/batch, loss=0.142]

...

Epoch 3/5: 105407batch [4:50:23,  6.05batch/s, loss=0.015]
Epoch 4/5: 83780batch [3:49:44,  4.74s/batch, loss=0.004]

Epoch 4 (Step 400000):
Train loss 0.248, Val loss 0.264
Train perplexity 1.281, Val perplexity 1.302


Epoch 4/5: 105407batch [4:48:59,  6.08batch/s, loss=0.011]
Epoch 5/5: 78373batch [3:34:43,  4.82s/batch, loss=0.055]

Epoch 5 (Step 500000):
Train loss 0.184, Val loss 0.238
Train perplexity 1.202, Val perplexity 1.268


Epoch 5/5: 105407batch [4:48:35,  6.09batch/s, loss=0.023]


In [36]:
torch.save(model.state_dict(), PROJECT_ROOT/'ml/notebooks/prot2text_model.pth')